# Visual-language assistant with MiniCPM-V 4.7 and OpenVINO

MiniCPM-V 4.7 combines a SigLIP vision encoder with a Qwen3.5 language model. Compared with [MiniCPM-V 4.6](../minicpm-v-4.6/minicpm-v-4.6.ipynb), it uses **canvas M-RoPE** to preserve the spatial relationship of image slices and the temporal order of video frames. See the [Transformers documentation](https://huggingface.co/docs/transformers/main/model_doc/minicpmv4_7).

This tutorial downloads a preconverted **1B dense** or **35B-A3B MoE** model by default. Clear the **Use preconverted model** checkbox to export a source checkpoint with [Optimum Intel](https://github.com/huggingface/optimum-intel/pull/2033) instead. Inference uses the [OpenVINO GenAI `VLMPipeline`](https://github.com/openvinotoolkit/openvino.genai/pull/4591) for text, single-image, multi-image, document, and video inputs. The 35B-A3B checkpoint requires substantially more memory; its GPU PagedAttention INT4 path also depends on [this OpenVINO fix](https://github.com/openvinotoolkit/openvino/pull/38631).

The [official model collection](https://huggingface.co/collections/openbmb/minicpm-v-47) includes [`openbmb/MiniCPM-V-4.7-1B`](https://huggingface.co/openbmb/MiniCPM-V-4.7-1B) and [`openbmb/MiniCPM-V-4.7-35B-A3B`](https://huggingface.co/openbmb/MiniCPM-V-4.7-35B-A3B). Preconverted OpenVINO models are available for [1B INT8](https://huggingface.co/OpenVINO/MiniCPM-V-4.7-1B-int8-ov), [1B INT4](https://huggingface.co/OpenVINO/MiniCPM-V-4.7-1B-int4-ov), [1B FP16](https://huggingface.co/OpenVINO/MiniCPM-V-4.7-1B-fp16-ov), [35B-A3B INT8](https://huggingface.co/OpenVINO/MiniCPM-V-4.7-35B-A3B-int8-ov), and [35B-A3B INT4](https://huggingface.co/OpenVINO/MiniCPM-V-4.7-35B-A3B-int4-ov). There is no preconverted 35B-A3B FP16 model; clear the checkbox to export that combination locally.

For local export, an empty `Export source:` field uses the selected checkpoint's local directory if present, or its published Hugging Face ID otherwise. Enter a local directory or a Hugging Face model ID to override it. OpenVINO GenAI handles multimodal preprocessing and generation for either path. Keep the model's visual downsampling configuration; do not override it with `4x`.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download or convert the model](#Download-or-convert-the-model)
- [Prepare the OpenVINO pipeline](#Prepare-the-OpenVINO-pipeline)
- [Text and single-image inference](#Text-and-single-image-inference)
- [Multiple images and document text](#Multiple-images-and-document-text)
- [Video inference](#Video-inference)
- [Optional DSpark speculative decoding](#Optional-DSpark-speculative-decoding)
- [Interactive image demo](#Interactive-image-demo)

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and uses development builds of OpenVINO GenAI and Optimum Intel. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/minicpm-v-4.7/minicpm-v-4.7.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

Downloading a preconverted model needs space for its OpenVINO weights; local export also needs disk space and RAM for the original weights. Run cells in order.

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Use Python 3.11–3.13. The installation pins Optimum Intel to [`f09a56ac`](https://github.com/huggingface/optimum-intel/commit/f09a56ace88149cd341233ddfa2bc018bff7c079), which includes MiniCPM-V 4.7 and the DSpark PagedAttention export contract, Transformers to 5.18.0, NNCF to 3.4.0, and the matching OpenVINO, Tokenizers, and GenAI nightly wheels dated 2026-11-16. These builds include the [DSpark runtime](https://github.com/openvinotoolkit/openvino.genai/pull/4790); stable 2026.4 packages do not provide this path.

This is an experimental dependency combination: the pinned Optimum Intel metadata declares `transformers<5.16`, although MiniCPM-V 4.7 export requires 5.18. Following the [upstream MiniCPM DSpark CI setup](https://github.com/openvinotoolkit/openvino.genai/blob/c6f9c59272851f89800754c63bf28bf365b5cd54/.github/workflows/test_matrices/windows/wheel_tests.yml), install its dependencies explicitly, then install the exporter with `--no-deps`. `pip check` will report this known metadata conflict; it is not a resolver-clean environment. Restart the kernel after installation if any of these packages were already imported.

In [ ]:
import urllib.request
from pathlib import Path

# Helpers come from `utils` of the cloned repository, otherwise from GitHub; local copies are overwritten
helper_files = ["notebook_utils.py", "cmd_helper.py"]
repo_utils = Path("../../utils")
in_repo = (repo_utils / "notebook_utils.py").is_file()
if in_repo:
    helper_contents = {helper_file: (repo_utils / helper_file).read_bytes() for helper_file in helper_files}
else:
    try:
        helper_contents = {}
        for helper_file in helper_files:
            url = f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{helper_file}"
            with urllib.request.urlopen(url, timeout=10) as response:
                helper_contents[helper_file] = response.read()
    except OSError as error:
        if any(not Path(helper_file).is_file() for helper_file in helper_files):
            raise
        print(f"Could not update helper files, using the existing copies: {error}")
        helper_contents = {}

for helper_file, content in helper_contents.items():
    Path(helper_file).write_bytes(content)

from notebook_utils import setup_notebook_venv

# Requirements installed below go to a notebook-specific virtual environment in the `.venv` folder
setup_notebook_venv()

In [ ]:
%pip install -q "torch>=2.1" torchvision "optimum~=2.3.0" "transformers==5.18.0" "nncf==3.4.0" "huggingface_hub>=0.23.2,<2.0" "requests>=2.33,<3.0" setuptools --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q --no-deps "git+https://github.com/huggingface/optimum-intel.git@f09a56ace88149cd341233ddfa2bc018bff7c079"
%pip install -q av Pillow ipywidgets "gradio>=6.29.1,<7"
%pip install -q --pre "openvino==2026.5.1.dev20261116" "openvino-tokenizers==2026.5.1.0.dev20261116" "openvino-genai==2026.5.1.0.dev20261116" --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly

from importlib.metadata import version

for package in ("optimum-intel", "transformers", "nncf", "openvino", "openvino-tokenizers", "openvino-genai"):
    print(f"{package}: {version(package)}")

In [ ]:
from pathlib import Path
import requests

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("minicpm-v-4.7.ipynb")

## Download or convert the model
[back to top ⬆️](#Table-of-contents:)

Choose **MiniCPM-V-4.7-1B** or **MiniCPM-V-4.7-35B-A3B** and the weight format below. By default, the corresponding preconverted `OpenVINO/<checkpoint>-<format>-ov` model is downloaded into `OpenVINO-<checkpoint>-ov/<format>`. Clear the **Use preconverted model** checkbox to export locally into `<model-name>-ov/<format>` instead. `Export source:` applies only to local export: leave it empty to use an existing checkpoint directory or `openbmb/<checkpoint>`, or enter a local directory or Hugging Face ID in `org/repo` form. The 35B-A3B download requires substantial disk space; local export additionally requires RAM for the original weights. Preconverted 35B-A3B FP16 is not published, but can be exported locally.

The [published 1B INT4 conversion](https://huggingface.co/OpenVINO/MiniCPM-V-4.7-1B-int4-ov/blob/7743dbc831ea3073795d4bbbf73490cdb0498808/openvino_config.json) uses group size 128 without calibration. A fresh local INT4 export of the canonical 1B checkpoint uses [tuned defaults](https://github.com/huggingface/optimum-intel/pull/2056): asymmetric group size 64, ratio 1.0, and scale estimation on GSM8K. These settings do not update a downloaded IR or an existing export. Use a new output directory to re-export; custom checkpoint names need their compression settings reviewed.

In [ ]:
import ipywidgets as widgets

checkpoint = widgets.Dropdown(
    options=["MiniCPM-V-4.7-1B", "MiniCPM-V-4.7-35B-A3B"],
    value="MiniCPM-V-4.7-1B",
    description="Checkpoint:",
)
model_source = widgets.Text(description="Export source:", placeholder="Local path or org/repo (optional)")
weight_format = widgets.Dropdown(options=["int8", "int4", "fp16"], value="int8", description="Weights:")
use_preconverted = widgets.Checkbox(value=True, description="Use preconverted model")
display(checkpoint, weight_format, use_preconverted, model_source)

In [ ]:
if use_preconverted.value:
    from huggingface_hub import snapshot_download

    if model_source.value.strip():
        raise ValueError("Export source is only used for local conversion. Clear 'Use preconverted model' first.")
    if checkpoint.value == "MiniCPM-V-4.7-35B-A3B" and weight_format.value == "fp16":
        raise ValueError("No preconverted 35B-A3B FP16 model is published. Clear 'Use preconverted model' to export it locally.")
    model_dir = Path(f"OpenVINO-{checkpoint.value}-ov") / weight_format.value
    model_id = f"OpenVINO/{checkpoint.value}-{weight_format.value}-ov"
    snapshot_download(repo_id=model_id, local_dir=model_dir)
else:
    from cmd_helper import optimum_cli
    import os
    import sys

    model_id = model_source.value.strip()
    if not model_id:
        model_id = checkpoint.value if Path(checkpoint.value).is_dir() else f"openbmb/{checkpoint.value}"
    if not Path(model_id).is_dir() and "/" not in model_id:
        raise FileNotFoundError(f"Provide a local checkpoint directory or a Hugging Face model ID (org/repo): {model_id}")
    model_dir = Path(f"{Path(model_id).name}-ov") / weight_format.value
    if not (model_dir / "openvino_language_model.xml").is_file():
        os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
        optimum_cli(
            model_id,
            model_dir,
            additional_args={"task": "image-text-to-text", "weight-format": weight_format.value},
        )
print(f"OpenVINO model: {model_dir}")

## Prepare the OpenVINO pipeline
[back to top ⬆️](#Table-of-contents:)

Optimum Intel exports the vision and language graphs and their configuration when local conversion is selected. OpenVINO GenAI loads either the downloaded or locally exported model into `VLMPipeline`, handles images and video, and supplies canvas M-RoPE positions. Keep each model's configuration and weights together.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

In [ ]:
import openvino_genai as ov_genai
from gradio_helper import load_model

pipe = load_model(str(model_dir.resolve()), device.value)
print(f"Loaded OpenVINO GenAI VLMPipeline from {model_dir} on {device.value}")

## Text and single-image inference
[back to top ⬆️](#Table-of-contents:)

The GenAI pipeline applies the model's chat template and adds image/video markers for inputs passed to `generate()`. Each call streams tokens to the cell. Thinking is disabled for short, comparable examples; set `enable_thinking=True` in `chat_history(...)` and increase `max_new_tokens` for a longer response.

In [ ]:
from io import BytesIO
from PIL import Image
import numpy as np
import openvino as ov


def load_image(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with Image.open(BytesIO(response.content)) as image:
        return image.convert("RGB")


def chat_history(question, *, enable_thinking=False):
    history = ov_genai.ChatHistory([{"role": "user", "content": question}])
    history.set_extra_context({"enable_thinking": enable_thinking})
    return history


def streamer(chunk):
    print(chunk, end="", flush=True)
    return False


cfg = ov_genai.GenerationConfig()
cfg.max_new_tokens = 512
print("Text only:", end=" ", flush=True)
_ = pipe.generate(chat_history("What is the capital of France? Answer in one word."), generation_config=cfg, streamer=streamer)

In [ ]:
cat = load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg")
display(cat)
_ = pipe.generate(chat_history("Describe this image."), images=[ov.Tensor(np.array(cat))], generation_config=cfg, streamer=streamer)

## Multiple images and document text
[back to top ⬆️](#Table-of-contents:)

The GenAI pipeline numbers separately supplied images when the exported model config enables image IDs. A document with small text exercises image slicing and the spatial arrangement of its patches; the synthetic invoice below needs no external demo asset. Its answer is illustrative, not a quality guarantee.

In [ ]:
bee = load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg")
_ = pipe.generate(
    chat_history("What is shown in the first image, and what is shown in the second?"),
    images=[ov.Tensor(np.array(cat)), ov.Tensor(np.array(bee))],
    generation_config=cfg,
    streamer=streamer,
)

In [ ]:
from PIL import ImageDraw

invoice = Image.new("RGB", (420, 220), "white")
draw = ImageDraw.Draw(invoice)
draw.text((20, 30), "INVOICE # OV-4721", fill="black")
draw.text((20, 80), "DUE DATE: 2026-11-15", fill="black")
draw.text((20, 130), "TOTAL: 125 USD", fill="black")
invoice = invoice.resize((1680, 880), Image.Resampling.NEAREST)
display(invoice)
_ = pipe.generate(
    chat_history("What are the invoice number and due date?"),
    images=[ov.Tensor(np.array(invoice))],
    generation_config=cfg,
    streamer=streamer,
)

## Video inference
[back to top ⬆️](#Table-of-contents:)

A video is **one temporal sequence**, not a set of individually numbered images. Sample up to eight frames for a manageable CPU example. PyAV avoids a system TorchCodec/FFmpeg dependency; the frames are already sampled, so pass their indices in `VideoMetadata` to avoid sampling again. Ask about event order to exercise temporal understanding.

In [ ]:
import av
import numpy as np
from notebook_utils import download_file

video_path = Path("football.mp4")
if not video_path.exists():
    download_file(
        "https://huggingface.co/datasets/openbmb/DemoCase/resolve/main/football.mp4",
        filename=video_path.name,
    )

frames = []
with av.open(str(video_path)) as container:
    fps = float(container.streams.video[0].average_rate or 1)
    next_second = 0
    for index, frame in enumerate(container.decode(video=0)):
        second = frame.time if frame.time is not None else index / fps
        if second < next_second:
            continue
        frames.append(frame.to_ndarray(format="rgb24"))
        next_second += 1
        if len(frames) == 8:
            break
if not frames:
    raise ValueError(f"No video frames decoded from {video_path}")
video_frames = np.stack(frames)
print(f"Selected {len(frames)} frames")

video_metadata = ov_genai.VideoMetadata()
video_metadata.fps = 1.0
video_metadata.frames_indices = list(range(len(frames)))
video_cfg = ov_genai.GenerationConfig()
video_cfg.max_new_tokens = 2048
_ = pipe.generate(
    chat_history("What happens in this video, in chronological order?"),
    videos=[ov.Tensor(video_frames)],
    videos_metadata=[video_metadata],
    generation_config=video_cfg,
    streamer=streamer,
)

## Optional DSpark speculative decoding
[back to top ⬆️](#Table-of-contents:)

[DSpark](https://github.com/openvinotoolkit/openvino.genai/blob/master/site/docs/concepts/optimization-techniques/speculative-decoding.mdx#dspark) uses a paired draft model with a Markov head to propose tokens that the target verifies. This optional text-only example uses the selected **Checkpoint** above with its own `-DSpark` draft: **1B INT8 target + INT4 draft**, or **35B-A3B INT4 target + INT4 draft**. Drafts are not standalone visual assistants. The regular examples and Gradio demo remain draft-free.

Enable the checkbox only when ready to download and export both original checkpoints and load an additional target/draft pair in memory. This section intentionally ignores **Export source**, **Weights**, and **Use preconverted model**: it exports canonical pairs with the pinned exporter into a separate `dspark-models` directory, including the target hidden-state metadata required by DSpark. October preconverted targets and older preview drafts must not be substituted without checking their metadata and export contract. An incomplete export raises an error; remove only that failed output directory before retrying.

The example uses CPU or GPU continuous batching, greedy decoding, one return sequence, and no prefix caching. NPU, beam search, adapters, and cache eviction are not supported for this path. GPU execution also requires the [PagedAttention masking fix](https://github.com/openvinotoolkit/openvino/pull/38897), included in the pinned nightly. Draft-token counts show that drafting was used; they do not prove an accuracy or speed improvement.

In [ ]:
enable_dspark = widgets.Checkbox(value=False, description="Run DSpark example", style={"description_width": "initial"})
enable_dspark

In [ ]:
if enable_dspark.value:
    import os
    import sys
    from cmd_helper import optimum_cli

    if device.value.split(".")[0] not in ("CPU", "GPU"):
        raise ValueError("The DSpark example requires CPU or GPU.")
    target_precision = {"MiniCPM-V-4.7-1B": "int8", "MiniCPM-V-4.7-35B-A3B": "int4"}[checkpoint.value]
    target_id = f"openbmb/{checkpoint.value}"
    draft_id = f"{target_id}-DSpark"
    draft_precision = "int4"
    pair_dir = Path("dspark-models") / f"{Path(target_id).name}-f09a56ac"
    target_dir = pair_dir / f"target-{target_precision}"
    draft_dir = pair_dir / f"draft-{draft_precision}"
    os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

    for source_id, output_dir, precision, task, graphs in (
        (
            target_id,
            target_dir,
            target_precision,
            "image-text-to-text",
            ("language_model", "text_embeddings_model", "vision_embeddings_model", "tokenizer", "detokenizer"),
        ),
        (draft_id, draft_dir, draft_precision, "text-generation-with-past", ("model", "markov_head_model")),
    ):
        if not output_dir.exists():
            optimum_cli(source_id, output_dir, additional_args={"task": task, "weight-format": precision})
        missing = [
            output_dir / f"openvino_{graph}.{ext}" for graph in graphs for ext in ("xml", "bin") if not (output_dir / f"openvino_{graph}.{ext}").is_file()
        ]
        if missing:
            raise FileNotFoundError(f"Incomplete DSpark export in {output_dir}: {missing}. Remove this failed output directory before retrying.")

    dspark_config = ov_genai.GenerationConfig()
    dspark_config.max_new_tokens = 128
    dspark_config.do_sample = False
    dspark_config.num_return_sequences = 1
    dspark_config.num_assistant_tokens = 7
    scheduler = ov_genai.SchedulerConfig()
    scheduler.enable_prefix_caching = False
    scheduler.num_linear_attention_blocks = dspark_config.num_assistant_tokens + 2
    draft = ov_genai.draft_model(str(draft_dir), device.value, scheduler_config=scheduler)
    dspark_pipe = ov_genai.VLMPipeline(str(target_dir), device.value, draft_model=draft, scheduler_config=scheduler)
    dspark_result = dspark_pipe.generate(chat_history("Write a Python function that checks whether a number is prime."), generation_config=dspark_config)
    print("DSpark response:", dspark_result)
    print("Draft tokens:", dspark_result.extended_perf_metrics.get_num_draft_tokens())
    print("Accepted tokens:", dspark_result.extended_perf_metrics.get_num_accepted_tokens())
    del dspark_pipe, draft
else:
    print("DSpark example is disabled; the regular pipeline is unchanged.")

## Interactive image demo
[back to top ⬆️](#Table-of-contents:)

After downloading or exporting once, run the next cell alone—even after restarting the kernel. Keep `gradio_helper.py` beside the notebook; select a downloaded or exported model/device and click an example row (preview, question, thinking). Responses stream, and thinking appears in a collapsible block. The model loads on the first request; if localhost is unavailable, Gradio creates a public `share=True` link.

In [ ]:
from gradio_helper import make_demo

demo = make_demo()
try:
    demo.launch(debug=True)
except ValueError as exc:
    if not str(exc).startswith("When localhost is not accessible, a shareable link must be created."):
        raise
    demo.launch(debug=True, share=True)
# If you are launching remotely, specify server_name and server_port
# EXAMPLE: `demo.launch(server_name='your server name', server_port='server port in int')`
# To learn more please refer to the Gradio docs: https://gradio.app/docs/